In [8]:
import json
import time
from pathlib import Path

import pandas as pd
import requests

SystemError: <class 'pandas._libs.lib.__pyx_defaults'> returned a result with an exception set

API CAI: 764 rifugi, 77 pagine da 10

In [ ]:
URL = "https://rifugi.cai.it/api/v1/shelters/search"

def fetch_page(page, per_page=None):
    """Scarica una pagina della ricerca e restituisce il blocco 'results'."""
    params = {"page": page}
    if per_page:
        params["per_page"] = per_page
    resp = requests.get(URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()["results"]

In [ ]:
shelters = []
page = 1

while True:
    results = fetch_page(page)

    # 1) add the shelters of this page to the list
    shelters.extend(results["data"])

    # 2) stop when this was the last page
    if results["current_page"] == results["last_page"]:
        break

    page += 1
    time.sleep(0.2)

print(len(shelters))

764


In [ ]:
Path("../data/raw").mkdir(parents=True, exist_ok=True)
with open("../data/raw/shelters.json", "w", encoding="utf-8") as f:
    json.dump(shelters, f, ensure_ascii=False)

In [ ]:
def flatten(shelter):
    """Turn one shelter into a flat dict: id_cai, title and one key per field."""
    row = {"id_cai": shelter["id_cai"], "title": shelter["title"]}
    for f in shelter["fields"]:
        row[f["name"]] = f["value"]
    return row

df = pd.DataFrame([flatten(s) for s in shelters])
print(df.shape)
df.head()

(764, 138)


,id_cai,title,branch_id,clustering_regione,clustering_provincia,migrate_id,alias,iso14001_certification,code_certification,category,...,camerate_da_4_posti_service,droughts_certification,type_management_property,fixedPhone_management_property,posti_letto_per_disabili_service,camerate_da_6_posti_service,tavolati_service,attivita_culturali_service,posti_attivita_culturali_service,greenCertification_certification
0,930010001,CAPANNA OSSERVATORIO RIFUGIO MARGHERITA,9300100,1,103,59c57aa189bf531d741f1ff3,Capanna Margherita,1,A119,E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,921200201,GIORDANO FELICE,9212002,2,7,59c57aa189bf531d741f1ff8,Bivacco Giordano,0,E167,E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,921601501,CITTA' DI GALLARATE,9216015,1,103,59c57aa189bf531d741f1ffd,Bivacco Città di Gallarate,0,E790,E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,920000102,CANZIO ETTORE,9200001,2,7,59c57aa189bf531d741f2016,Bivacco Canzio,0,D012,E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,921201901,ROSSI GIORGIO/VOLANTE CESARE,9212019,2,7,59c57aa189bf531d741f201b,Bivacco Rossi e Volante,0,A094,E,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
cols = ["region_geo", "ownerRegion_geo", "municipality_geo",
        "massif_geo", "locality_geo", "altitude_geo"]
print(df[cols].notna().sum())

print("regione diversa da regione proprietaria:",
      (df["region_geo"] != df["ownerRegion_geo"]).sum())
print("comuni distinti:", df["municipality_geo"].nunique())
print("massicci distinti:", df["massif_geo"].nunique())

region_geo          764
ownerRegion_geo     745
municipality_geo    762
massif_geo          756
locality_geo        757
altitude_geo        764
dtype: int64
regione diversa da regione proprietaria: 85
comuni distinti: 414
massicci distinti: 68


In [ ]:
per_mun = df["municipality_geo"].value_counts()
print((per_mun == 1).sum(), "comuni con un solo rifugio su", len(per_mun))
print(per_mun.head(10))

print(df["massif_geo"].value_counts().describe())

both = df.dropna(subset=["region_geo", "ownerRegion_geo"])
diff = both[both["region_geo"] != both["ownerRegion_geo"]]
print(len(diff))
print(diff[["title", "region_geo", "ownerRegion_geo"]].head(10))

service_cols = [c for c in df.columns if c.endswith("_service")]
print(len(service_cols))
print(df[service_cols].notna().sum().sort_values(ascending=False).head(15))

273 comuni con un solo rifugio su 414
municipality_geo
Courmayeur              18
Macugnaga               13
Alagna Valsesia         12
Auronzo di Cadore       12
Valfurva                 9
Val Masino               9
Valdieri                 9
Tarvisio                 9
Cogne                    8
Chiesa in Valmalenco     7
Name: count, dtype: int64
count     68.000000
mean      11.117647
std       19.665508
min        1.000000
25%        1.000000
50%        3.000000
75%       11.000000
max      118.000000
Name: count, dtype: float64
66
                                        title           region_geo  \
2                         CITTA' DI GALLARATE             Piemonte   
7                                MUSSO BIAGIO             Svizzera   
11  SELLA QUINTINO al Felik (NUOVO E VECCHIO)        Valle d'Aosta   
13                            CITTA' DI LUINO             Piemonte   
16                           CITTA' DI CANTU'  Trentino-Alto Adige   
19                              BALZOL

In [ ]:
print(df("municipality_geo").head(20))


NameError: name 'df' is not defined